# 🎙️ AI Video Studio: RVC Cloud Model Trainer
This notebook is specifically designed to train your custom AI voice models (e.g., Mythia, Furina, or your own voice) using Google Colab's cloud GPUs for free, fast (~15 mins), and stable execution.

---

### 📁 Step 1: Connect to Your Google Drive (Highly Recommended)
Google Colab instances are temporary. By mounting your Google Drive, your voice datasets and trained voice models (`.pth` and `.index` files) will be saved permanently and will not be lost when the session closes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('[SUCCESS] Google Drive mounted successfully!')

### ⚙️ Step 2: Configure Stable Python 3.10 Virtual Environment (Crucial)
Google Colab currently runs **Python 3.12** by default, which is incompatible with RVC's core dependencies (`numba`, `fairseq`). This cell cleans up any previous system path overrides, installs a clean **Python 3.10** binary alongside, and sets up a fully isolated **Virtual Environment (`/content/venv`)** that inherits Colab's pre-installed PyTorch GPU suites to ensure instant, stable setup.

In [ ]:
print('[INFO] Auto-repairing any corrupted system python overrides...')
!sudo update-alternatives --remove-all python3 2>/dev/null || true

print('[INFO] Installing Python 3.10 system binaries...')
!sudo apt-get update -y
!sudo apt-get install python3.10 python3.10-dev python3.10-venv python3.10-distutils -y

print('[INFO] Creating Python 3.10 Virtual Environment (inheriting system PyTorch)...')
!python3.10 -m venv --system-site-packages /content/venv

print('[INFO] Installing compatible pip manager (pip<24.1) inside virtual environment...')
!/content/venv/bin/python -m pip install --upgrade "pip<24.1"

print('\n[SUCCESS] Python 3.10 Virtual Environment configured successfully!')
print('Active Python path: /content/venv/bin/python')
!/content/venv/bin/python --version

### 📦 Step 3: Clone RVC & Install Stable Dependencies
This cell clones the RVC WebUI and installs the necessary machine learning libraries into our isolated **Python 3.10 virtual environment**. It has a built-in **fail-safe** to automatically configure the virtual environment if Step 2 was skipped.

In [ ]:
# 0. Fail-Safe: Automatically set up venv if Step 2 was skipped
print('[INFO] Checking virtual environment status...')
!if [ ! -d "/content/venv" ]; then \
    echo "[WARNING] Virtual Environment not found! Setting up Python 3.10 automatically..."; \
    sudo apt-get update -y && \
    sudo apt-get install python3.10 python3.10-dev python3.10-venv python3.10-distutils -y && \
    python3.10 -m venv --system-site-packages /content/venv && \
    /content/venv/bin/python -m pip install --upgrade "pip<24.1"; \
fi

# 1. Clean old directory if exists
%cd /content
!rm -rf /content/Retrieval-based-Voice-Conversion-WebUI

# 2. Clone the stable v1.0 RVC WebUI repository
print('[INFO] Cloning RVC WebUI repository...')
!git clone -b v1.0 https://github.com/camenduru/Retrieval-based-Voice-Conversion-WebUI.git

# 3. Enter directories
%cd /content/Retrieval-based-Voice-Conversion-WebUI

# 4. Lock in stable NumPy and SciPy first (Crucial to prevent NumPy 2.x conflicts and compilation errors)
print('[INFO] Installing stable legacy core (numpy 1.23.5, scipy 1.9.3, Cython)...')
!/content/venv/bin/python -m pip install Cython numpy==1.23.5 scipy==1.9.3

# 5. Install Fairseq (Using MiroPsota index for instant wheel, with compile fallback)
print('[INFO] Installing fairseq from precompiled matching wheel...')
!/content/venv/bin/python -m pip install fairseq --extra-index-url https://miropsota.github.io/torch_packages_builder || \
 /content/venv/bin/python -m pip install fairseq==0.12.2 || \
 /content/venv/bin/python -m pip install git+https://github.com/pytorch/fairseq.git

# 6. Install Numba & Librosa (Will now install instantly from pre-compiled wheels!)
print('[INFO] Installing stable voice libraries (numba, librosa)...')
!/content/venv/bin/python -m pip install numba==0.56.4 librosa==0.9.2

# 7. Install standard supporting libraries
print('[INFO] Installing pure-python support packages...')
!/content/venv/bin/python -m pip install gradio gdown mega.py tensorboardX ffmpeg ffmpeg-python

# 8. Install Scientific and compiled packages
print('[INFO] Installing scientific suites (faiss-cpu, parselmouth, pyworld)...')
!/content/venv/bin/python -m pip install faiss-cpu praat-parselmouth pyworld

# 9. Download pre-trained Hubert base model
print('[INFO] Downloading pre-trained hubert_base.pt...')
!wget https://huggingface.co/audo/VoiceConversionWebUI/resolve/main/hubert_base.pt -O hubert_base.pt

# 10. Create weights and logs folders
!mkdir -p /content/Retrieval-based-Voice-Conversion-WebUI/weights
!mkdir -p /content/Retrieval-based-Voice-Conversion-WebUI/logs

print('\n[SUCCESS] All RVC dependencies and base models installed successfully under Python 3.10!')

### 🚀 Step 4: Launch RVC WebUI (Gradio Tunnel)
Run this cell to start the RVC WebUI server on Google Colab using our custom Python 3.10 virtual environment.

Once the startup log finishes, look for a blue link that says:
👉 **`Running on public URL: https://xxxxxxxx.gradio.live`**

**Click that public Gradio link** to open the RVC Studio interface in your browser and start training!

In [ ]:
%cd /content/Retrieval-based-Voice-Conversion-WebUI
print('[INFO] Starting RVC Server... Please wait for the public Gradio link to appear below.')
!/content/venv/bin/python infer-web.py --colab --share